# Agent Architectures in Practice — Multi‑Framework Lab (Fixed)
This version is organized into clear sections with code that **runs** (with graceful fallbacks).  
If you haven't installed the optional libraries yet, the cells will skip those sections and explain what to install.

## 0) Setup

**Set up the environment for the multi-framework lab.**

- The first three lines are commented install commands. Uncomment them if the required packages are not installed, then restart the kernel.
- `load_dotenv(override=True)` reads environment variables from `.env`.
- `OPENAI_API_KEY` is read from the environment. If it is missing, a tip is printed so you know why OpenAI-backed cells may fail.


In [1]:
# Install (uncomment if needed). Run this cell, then restart the kernel.
# Note: package names are intentionally explicit to avoid version confusion.
# !pip install -U langchain langchain-openai langchain-community crewai pyautogen chromadb

import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Read API keys if available (optional for sections that use OpenAI)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("Tip: Set OPENAI_API_KEY in your environment if you want to use OpenAI-backed sections.")

## 2) LangChain — Tool‑Using Agent (Zero‑Shot ReAct)

**Install the tokenizer dependency used by some LangChain/OpenAI utilities.**

- `tiktoken` is OpenAI's fast byte-pair encoding tokenizer.
- LangChain and OpenAI models may need it for token counting or prompt formatting.


In [4]:
!pip install tiktoken

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/918.7 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/918.7 kB 1.6 MB/s eta 0:00:01
   ---------------------------------- ----- 786.4/918.7 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 918.7/918.7 kB 1.2 MB/s  0:00:00


**Build a LangChain zero-shot ReAct agent with two tools.**

What the code does, line by line:
- Imports `create_agent`, `Tool`, and `ChatOpenAI`.
- Defines `safe_calculate`: a guarded arithmetic evaluator using `eval()`.
- Defines `search_corpus`: a tiny keyword lookup over agentic-AI topics.
- Wraps both functions as `Tool` objects with descriptive names and docstrings.
- Creates a `gpt-4o-mini` chat model with `temperature=0` for reproducible reasoning.
- Calls `create_agent(model, tools)` to produce the runnable agent.
- Invokes the agent with the question `"What is 8% of 20000?"` and prints the full response.

Expected output:
- The agent thinks step-by-step (ReAct) and calls the `calculator` tool.
- The printed `response` shows the message history, including the tool call and the final answer.


In [2]:
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI

# ---- Define tools ----
def safe_calculate(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

def search_corpus(query: str) -> str:
    corpus = {
        "langgraph": "LangGraph is used for stateful workflows, retries, and human-in-the-loop systems.",
        "crewai": "CrewAI focuses on role-based multi-agent collaboration.",
        "autogen": "AutoGen enables multi-agent conversations."
    }
    query = query.lower()
    for key, value in corpus.items():
        if key in query:
            return value
    return "No relevant results found."

tools = [
    Tool(
        name="calculator",
        func=safe_calculate,
        description="Use this to perform mathematical calculations."
    ),
    Tool(
        name="search",
        func=search_corpus,
        description="Searches a local knowledge corpus for information."
    )
]

# ---- Model ----
model = ChatOpenAI(
    model="gpt-4o-mini",  # or any supported model
    temperature=0
)

# ---- Create agent (modern API) ----
agent = create_agent(
    model=model,
    tools=tools
)

# ---- Run agent ----
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 8% of 20000?"}
    ]
})

print(response)

C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


{'messages': [HumanMessage(content='What is 8% of 20000?', additional_kwargs={}, response_metadata={}, id='448d2e6a-7b24-4652-be70-58022a9caa30'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 88, 'total_tokens': 110, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2b4c33dc35', 'id': 'chatcmpl-DgOJaVV8PwMuXAnNmcaViKZ8t2CI5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3466-c413-7052-ad29-5124179ad73f-0', tool_calls=[{'name': 'calculator', 'args': {'__arg1': '0.08 * 20000'}, 'id': 'call_ULT29O6MEryrV6AD1Lc05Ys3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 88, 'output_t

**Ask the agent a second question about an agentic-AI topic.**

- The user asks `"What is crewai"`.
- The agent should call the `search` tool, retrieve the CrewAI description, and formulate a concise answer.


In [16]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is crewai"}
    ]
})


**Display the full response object.**

- In Jupyter, writing `response` at the end of a cell pretty-prints the object.
- You can inspect the message list to see exactly when the agent decided to call a tool.


In [17]:
response

{'messages': [HumanMessage(content='What is crewai', additional_kwargs={}, response_metadata={}, id='3d7c50dd-7bda-40f1-8f3a-98924386b097'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 82, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2b4c33dc35', 'id': 'chatcmpl-DgMwGocjUrBN4TNA29q94J4Pc2Zyj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3416-19c0-7553-bde8-052f86ecd1c1-0', tool_calls=[{'name': 'search', 'args': {'__arg1': 'crewai'}, 'id': 'call_V4E07LqgrbXiEkA6pQhZeD1w', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 16, 'to

## 3) AutoGen — Two‑Agent Chat

**Install the base AutoGen package.**

- `pyautogen` is the classic AutoGen SDK.
- We install it before using the newer `autogen-agentchat` components.


In [19]:
!pip install pyautogen

Defaulting to user installation because normal site-packages is not writeable


**Install the modern AutoGen agent-chat and OpenAI extension packages.**

- `autogen-agentchat` provides high-level conversational agents.
- `autogen-ext[openai]` adds the OpenAI model client so agents can call GPT models.


In [22]:
!pip install -U "autogen-agentchat" "autogen-ext[openai]"

Defaulting to user installation because normal site-packages is not writeable


**Import the AutoGen agent classes.**

- `AssistantAgent` is an LLM-backed agent that can generate replies.
- `UserProxyAgent` is an agent that can stand in for a human user (imported for completeness).


In [23]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

**Set up a two-agent team and a termination condition.**

What the code does, line by line:
- Imports asyncio and AutoGen chat building blocks.
- Creates an `OpenAIChatCompletionClient` pointing to `gpt-4o-2024-08-06`.
- Defines `primary_agent` with the system message `"You are a helpful AI assistant."`
- Defines `critic_agent` whose job is to give feedback and say `APPROVE` when satisfied.
- Creates a `TextMentionTermination` condition that stops the chat when `APPROVE` appears.
- Builds a `RoundRobinGroupChat` team containing both agents.

Why this matters:
- Two agents collaborating can produce higher-quality output than a single model call.
- The termination condition keeps the conversation from running forever.


In [26]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import ExternalTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Create an OpenAI model client.
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-2024-08-06",
    # api_key="sk-...", # Optional if you have an OPENAI_API_KEY env variable set.
)

# Create the primary agent.
primary_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    system_message="You are a helpful AI assistant.",
)

# Create the critic agent.
critic_agent = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="Provide constructive feedback. Respond with 'APPROVE' to when your feedbacks are addressed.",
)

# Define a termination condition that stops the task if the critic approves.
text_termination = TextMentionTermination("APPROVE")

# Create a team with the primary and critic agents.
team = RoundRobinGroupChat([primary_agent, critic_agent], termination_condition=text_termination)





**Run the team with a creative writing task.**

- `await team.run(task="Write a short poem about the fall season.")` starts the round-robin chat.
- The primary agent drafts a poem; the critic reviews it.
- The chat ends when the critic responds with `APPROVE`.
- `print(result)` shows the final task result, including the conversation log.


In [27]:
# Use `asyncio.run(...)` when running in a script.
result = await team.run(task="Write a short poem about the fall season.")
print(result)


messages=[TextMessage(id='e6d7b81a-e56c-4013-a01b-0ec975426a7a', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 17, 4, 6, 27, 416046, tzinfo=datetime.timezone.utc), content='Write a short poem about the fall season.', type='TextMessage'), TextMessage(id='aa4c2bbc-af34-443a-80d7-a081069d8af5', source='primary', models_usage=RequestUsage(prompt_tokens=28, completion_tokens=109), metadata={}, created_at=datetime.datetime(2026, 5, 17, 4, 6, 30, 989825, tzinfo=datetime.timezone.utc), content="Leaves cascade in vibrant hues,  \nAmber, gold, and crimson fuse.  \nWhispering winds in gentle sweep,  \nHarvest moons and nights that creep.  \n\nPumpkin patches, apple spice,  \nCrisp air’s kiss, a cool entice.  \nSweaters warm as daylight fades,  \nIn fall’s embrace, the world cascades.  \n\nNature's canvas, bold and grand,  \nAutumn paints with gentle hand.  \nA fleeting dance, this season's call—  \nA wondrous, fleeting rise and fall.  ", type='TextMessage'),

## 4) (Optional) LangGraph Sketch — Pseudo‑code

**Sketch a LangGraph workflow in pseudo-code.**

- LangGraph lets you build stateful, cyclic graphs of agents.
- The sketch shows three nodes: `start`, `search`, and `reflect`.
- Edges connect `start -> search -> reflect`, and `reflect` can loop back to `search` for retries.
- This cell is commented out so it does not execute; it is meant as a conceptual blueprint.


In [ ]:
# Pseudo-code only (no execution):
# from langgraph.graph import StateGraph
# graph = StateGraph()
# graph.add_node("start", StartNode())
# graph.add_node("search", SearchNode())
# graph.add_node("reflect", ReflectNode())
# graph.set_entry_point("start")
# graph.add_edge("start", "search")
# graph.add_edge("search", "reflect")
# graph.add_edge("reflect", "search")  # retry loop

---

**Summary**

This notebook demonstrated three ways to build multi-agent systems:
1. **LangChain ReAct agent** — a single LLM that chooses between math and search tools.
2. **AutoGen team** — two agents (primary + critic) that converse until a termination condition is met.
3. **LangGraph sketch** — a stateful graph where nodes can loop back for retries and reflection.
